In [ ]:
# Imports
import numpy as np
import torch
from pettingzoo.mpe import simple_reference_v3
from TorchRL_MAC_utils import default_cfg, make_env, get_agent_names, build_modules, collect_rollout

## 1. Native PettingZoo Parallel Mode

In [ ]:
# Create environment
pz_env = simple_reference_v3.parallel_env(max_cycles=25, continuous_actions=False)

print("Agents:", pz_env.possible_agents)
print("\nObservation spaces:")
for agent in pz_env.possible_agents:
    print(f"  {agent}: {pz_env.observation_space(agent)}")
print("\nAction spaces:")
for agent in pz_env.possible_agents:
    print(f"  {agent}: {pz_env.action_space(agent)}")

In [ ]:
# Reset
observations = pz_env.reset(seed=42)

print("Reset returns dict of observations:")
for agent, obs in observations.items():
    print(f"  {agent}: shape={obs.shape}, dtype={obs.dtype}")

In [ ]:
# Single step with random actions
actions = {agent: pz_env.action_space(agent).sample() for agent in pz_env.possible_agents}
print("Actions:", actions)

obs, rewards, dones, truncations, infos = pz_env.step(actions)

print("\nStep returns:")
print("  Observations:", {k: v.shape for k, v in obs.items()})
print("  Rewards:", rewards)
print("  Dones:", dones)
print("  Truncations:", truncations)
print("  Team reward:", sum(rewards.values()))

## 2. TorchRL Wrapper

In [ ]:
# Create TorchRL-wrapped environment
try:
    from torchrl.envs.libs.pettingzoo import PettingZooEnv
    
    torchrl_env = PettingZooEnv(
        task="mpe/simple_reference_v3",
        parallel=True,
        categorical_actions=True,
        seed=42,
        done_on_any=False,
        max_cycles=25,
        continuous_actions=False
    )
    
    print("✓ TorchRL wrapper created")
    print(f"Type: {type(torchrl_env)}")
    
    # Access underlying PettingZoo env
    if hasattr(torchrl_env, '_env'):
        base_env = torchrl_env._env
    elif hasattr(torchrl_env, 'env'):
        base_env = torchrl_env.env
    else:
        base_env = torchrl_env
    
    print(f"Underlying env: {type(base_env)}")
    print(f"Agents: {base_env.possible_agents}")
    
except Exception as e:
    print(f"TorchRL wrapper not available or error: {e}")
    print("Note: mac_utils.py handles this gracefully with fallbacks")

## 3. MAC Wrapper Layer

In [ ]:
# Create config
cfg = default_cfg()
cfg.num_envs = 2
cfg.rollout_len = 4
cfg.device = 'cpu'

print("Config created:")
print(f"  num_envs={cfg.num_envs}, rollout_len={cfg.rollout_len}")
print(f"  hidden_dim={cfg.hidden_dim}, lr={cfg.lr}")

In [ ]:
# Make environment
env = make_env(cfg)
agents = get_agent_names(env)

print("Environment created:")
print(f"  Agents: {agents}")

In [ ]:
# Build modules
actors, critic, specs = build_modules(cfg, env)

print("\nModules built:")
print(f"  Actors: {list(actors.keys())}")
print(f"  Critic: {type(critic).__name__}")
print(f"\nSpecs:")
print(f"  Observation dims: {specs['obs_dims']}")
print(f"  Action dims: {specs['action_dims']}")
print(f"  Action types: {specs['action_types']}")

# Count parameters
total_params = sum(p.numel() for p in critic.parameters())
for actor in actors.values():
    total_params += sum(p.numel() for p in actor.parameters())
print(f"\nTotal parameters: {total_params:,}")

In [ ]:
# Collect a small rollout
batch = collect_rollout(cfg, env, actors, critic, device=cfg.device)

print("\nRollout batch collected:")
print(f"  Transitions: {batch['reward'].shape[0]} (= num_envs × rollout_len)")
print(f"\nBatch keys: {list(batch.keys())}")
print(f"\nBatch shapes:")
for key, value in batch.items():
    if key in ['obs', 'actions', 'logp', 'entropy']:
        print(f"  {key}:")
        for agent, tensor in value.items():
            print(f"    {agent}: {tensor.shape}")
    else:
        print(f"  {key}: {value.shape}")

In [ ]:
# Inspect batch content
print("Sample batch content:")
print(f"  Rewards (first 4): {batch['reward'][:4].squeeze().tolist()}")
print(f"  Dones (first 4): {batch['done'][:4].squeeze().tolist()}")
if 'values' in batch:
    print(f"  Values (first 4): {batch['values'][:4].squeeze().tolist()}")
print(f"\n  Mean reward: {batch['reward'].mean().item():.3f}")
print(f"  Mean entropy: {sum(batch['entropy'][agent].mean().item() for agent in agents):.3f}")

## Summary

This notebook demonstrated:
1. **PettingZoo parallel API**: `reset()` returns obs dict, `step(actions)` returns 5-tuple
2. **TorchRL wrapper**: Wraps PettingZoo for TensorDict-based interaction
3. **MAC wrapper**: High-level API for config, env creation, model building, and rollout collection

The MAC wrapper abstracts away environment details and provides a clean interface for training and evaluation.